# Acne Lesion Segmentation & Severity Classification
**Dataset:** ACNE04-v2 | **Model:** Hybrid ResNet-50 + Dilated Decoder + AMFM

---
## Urutan Menjalankan
Jalankan cell **dari atas ke bawah secara berurutan**. Jangan skip cell apapun.

| # | Cell | Waktu |
|---|---|---|
| 1 | Setup GPU & Import | ~30 detik |
| 2 | Mount Google Drive | ~10 detik |
| 3 | Install Dependencies | ~1 menit |
| 4 | Import Python Libraries | ~5 detik |
| 5 | Download Dataset dari Folder Drive | ~3 menit |
| 6 | Extract & Auto-detect Path | ~2 menit |
| 7 | Ekstrak Scripts dari Drive | ~30 detik |
| 8 | Konfigurasi & Validasi Path | ~5 detik |
| 9 | Generate Refined Mask | ~5-15 menit |
| 10 | Training | ~2-4 jam |
| 11 | Inference & Visualisasi | ~10 menit |


---
## 1. Setup GPU
> **Aktifkan GPU dulu sebelum menjalankan notebook:**  
> `Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save`

In [ ]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU name        :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print()
    print('⚠️  GPU tidak aktif!')
    print('   Aktifkan: Runtime → Change runtime type → T4 GPU')

---
## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive berhasil di-mount di /content/drive')

---
## 3. Install Dependencies

In [ ]:
%%capture
!pip install torch torchvision pillow opencv-python-headless numpy matplotlib pandas tqdm scikit-learn gdown

---
## 4. Import Python Libraries
> **Wajib dijalankan sebelum cell apapun di bawah ini.**

In [ ]:
import os
import sys
import json
import shutil
import random
import tarfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch

print('✅ Semua library berhasil diimport')
print(f'   Python  : {sys.version.split()[0]}')
print(f'   PyTorch : {torch.__version__}')
print(f'   Device  : {"cuda" if torch.cuda.is_available() else "cpu"}')

---
## 5. Download Dataset ACNE04-v2 dari Folder Google Drive

Dataset diambil dari folder Drive publik:

`https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ`


In [ ]:
# ══════════════════════════════════════════════════════════════
# Dataset dari folder Google Drive publik
# ══════════════════════════════════════════════════════════════

import gdown

DATASET_DRIVE_FOLDER = 'https://drive.google.com/drive/folders/18yJcHXhzOv7H89t-Lda6phheAicLqMuZ'
DOWNLOAD_DIR = Path('/content/acne_images')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

gdown.download_folder(
    DATASET_DRIVE_FOLDER,
    output=str(DOWNLOAD_DIR),
    quiet=False,
    use_cookies=False,
)

print('✅ Dataset selesai didownload ke', DOWNLOAD_DIR)


---
## 6. Extract Tar & Auto-detect Path Dataset

Folder Drive dataset berisi archive seperti `Classification.tar` dan `Detection.tar`.
Cell berikut akan extract semua archive itu, lalu cell setelahnya mencari folder gambar dan annotation JSON otomatis.


In [ ]:
# Extract semua archive tar yang ditemukan di DOWNLOAD_DIR
archive_patterns = ['*.tar', '*.tar.gz', '*.tgz']
processed_archives = set()

while True:
    tar_files = []
    for pattern in archive_patterns:
        tar_files.extend(DOWNLOAD_DIR.rglob(pattern))
    tar_files = sorted(p for p in tar_files if p not in processed_archives)

    if not tar_files:
        break

    for tar_path in tar_files:
        extract_to = tar_path.parent
        print(f'Extracting {tar_path.name} → {extract_to}')
        with tarfile.open(tar_path) as tf:
            tf.extractall(path=extract_to)
        processed_archives.add(tar_path)

if processed_archives:
    print(f'✅ {len(processed_archives)} archive berhasil diekstrak')
else:
    print('Tidak ada file .tar/.tar.gz/.tgz ditemukan — mungkin dataset sudah berupa folder langsung.')

print()
print('Struktur folder setelah download/extract (maks. 120 item):')
all_paths = sorted(DOWNLOAD_DIR.rglob('*'))
for p in all_paths[:120]:
    indent = '  ' * (len(p.relative_to(DOWNLOAD_DIR).parts) - 1)
    print(f'{indent}  {p.name}{"/ " if p.is_dir() else ""}')
if len(all_paths) > 120:
    print(f'... {len(all_paths) - 120} item lain disembunyikan')


In [ ]:
# Auto-detect folder gambar dan annotation JSON/XML

def find_image_dir(root):
    best_dir, best_count = None, 0
    for d in root.rglob('*'):
        if d.is_dir():
            count = len(list(d.glob('*.jpg'))) + len(list(d.glob('*.JPG'))) + len(list(d.glob('*.jpeg'))) + len(list(d.glob('*.JPEG')))
            if count > best_count:
                best_dir, best_count = d, count
    return best_dir

def find_annotation_path(root):
    json_candidates = list(root.rglob('*.json'))
    for c in json_candidates:
        if 'annot' in c.name.lower() or 'acne04' in c.name.lower():
            return c
    if json_candidates:
        return json_candidates[0]

    xml_dirs = []
    for d in root.rglob('*'):
        if d.is_dir():
            n_xml = len(list(d.glob('*.xml')))
            if n_xml > 0:
                xml_dirs.append((n_xml, d))
    if xml_dirs:
        xml_dirs.sort(reverse=True)
        return xml_dirs[0][1]
    return None

DETECTED_IMAGE_DIR = find_image_dir(DOWNLOAD_DIR)
DETECTED_ANNOT_JSON = find_annotation_path(DOWNLOAD_DIR)

print('=== AUTO-DETECT RESULT ===')
if DETECTED_IMAGE_DIR:
    n_img = len(list(DETECTED_IMAGE_DIR.glob('*.jpg'))) + len(list(DETECTED_IMAGE_DIR.glob('*.JPG'))) + len(list(DETECTED_IMAGE_DIR.glob('*.jpeg'))) + len(list(DETECTED_IMAGE_DIR.glob('*.JPEG')))
    print(f'✅ Image dir  : {DETECTED_IMAGE_DIR}')
    print(f'   Jumlah img : {n_img}')
else:
    raise FileNotFoundError('Image dir tidak ditemukan di hasil download Drive. Cek isi Classification.tar.')

if DETECTED_ANNOT_JSON:
    if DETECTED_ANNOT_JSON.is_dir():
        print(f'✅ Annotation : {DETECTED_ANNOT_JSON} ({len(list(DETECTED_ANNOT_JSON.glob("*.xml")))} XML)')
    else:
        print(f'✅ Annotation : {DETECTED_ANNOT_JSON}')
else:
    raise FileNotFoundError('Annotation tidak ditemukan. Detection.tar harus berisi JSON atau folder XML Annotations.')


---
## 7. Setup Scripts Project dari Google Drive

Jalankan cell di bawah untuk mengekstrak `MyDrive/acne_project/acne_scripts.zip`.


In [ ]:
# ══════════════════════════════════════════════════════════════
# Scripts dari Google Drive sendiri
#
# Struktur yang wajib ada di Drive:
#   MyDrive/acne_project/acne_scripts.zip
# ══════════════════════════════════════════════════════════════

SCRIPTS_ZIP = Path('/content/drive/MyDrive/acne_project/acne_scripts.zip')
SCRIPTS_ROOT = Path('/content/acne_project')
SCRIPTS_ROOT.mkdir(parents=True, exist_ok=True)

if not SCRIPTS_ZIP.exists():
    raise FileNotFoundError(
        'acne_scripts.zip tidak ditemukan. Upload ke MyDrive/acne_project/acne_scripts.zip'
    )

!unzip -q -o "{SCRIPTS_ZIP}" -d "{SCRIPTS_ROOT}/"

script_candidates = list(SCRIPTS_ROOT.rglob('refine_masks_color.py'))
if not script_candidates:
    raise FileNotFoundError('refine_masks_color.py tidak ditemukan di acne_scripts.zip')

SCRIPTS_DIR = script_candidates[0].parent
print('✅ Scripts dir:', SCRIPTS_DIR)
print('Exists:', SCRIPTS_DIR.exists())


In [ ]:
# Tambahkan scripts ke Python path
for p in [str(SCRIPTS_DIR), str(SCRIPTS_DIR / 'code_dari_claude')]:
    if p not in sys.path:
        sys.path.insert(0, p)

py_files = list(SCRIPTS_DIR.glob('*.py'))
print(f'✅ {len(py_files)} script Python ditemukan di {SCRIPTS_DIR}')
for f in sorted(py_files):
    print(f'   {f.name}')

---
## 8. Konfigurasi Path

> Biasanya tidak perlu diubah jika pakai auto-detect di atas.  
> Ubah manual hanya jika path salah.

In [ ]:
# ============================================================
# KONFIGURASI TERPUSAT — semua path & parameter ada di sini
# ============================================================

# --- INPUT ---
IMAGE_DIR  = DETECTED_IMAGE_DIR    # dari auto-detect
ANNOT_JSON = DETECTED_ANNOT_JSON   # JSON atau folder XML dari auto-detect

# Override manual jika auto-detect salah:
# IMAGE_DIR  = Path('/content/acne_images/ACNE04/Classification/small_1024')
# ANNOT_JSON = Path('/content/acne_images/Detection/VOC2007/Annotations')

# --- OUTPUT (Drive, agar tidak hilang saat sesi berakhir) ---
OUTPUT_ROOT = Path('/content/drive/MyDrive/acne_project/output')
MASK_DIR    = OUTPUT_ROOT / 'refined_masks_output'
TRAIN_DIR   = OUTPUT_ROOT / 'training_runs'
INFER_DIR   = OUTPUT_ROOT / 'predictions'

for d in [OUTPUT_ROOT, MASK_DIR, TRAIN_DIR, INFER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Device & manifest ---
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
REFINED_MANIFEST = MASK_DIR / 'refined_manifest.csv'

# --- Script paths ---
REFINE_SCRIPT  = SCRIPTS_DIR / 'refine_masks_color.py'
TRAIN_SCRIPT_A = SCRIPTS_DIR / 'train_proposal_hybrid_resnet50_malds.py'
TRAIN_SCRIPT_B = SCRIPTS_DIR / 'code_dari_claude' / 'train_hybrid_resnet50_staged.py'
INFER_SCRIPT   = SCRIPTS_DIR / 'infer_testing_images.py'

# --- Output run directories ---
OUTPUT_A  = TRAIN_DIR / 'proposal_hybrid_resnet50_malds'
OUTPUT_B  = TRAIN_DIR / 'hybrid_resnet50_staged'
INFER_OUT = INFER_DIR / 'proposal_hybrid_resnet50_malds'

# --- Training hyperparameters ---
PHASE1_EPOCHS = 30   # frozen backbone
PHASE2_EPOCHS = 90   # full finetune
BATCH_SIZE    = 16   # T4 VRAM ~15GB, aman untuk image 320px
IMAGE_SIZE    = 320
NUM_WORKERS   = 4

print('=== PATH CONFIGURATION ===')
print(f'Image dir      : {IMAGE_DIR}')
print(f'Annotation     : {ANNOT_JSON}')
print(f'Scripts        : {SCRIPTS_DIR}')
print(f'Output         : {OUTPUT_ROOT}')
print(f'Device         : {DEVICE}')
print(f'Batch / ImgSz  : {BATCH_SIZE} / {IMAGE_SIZE}px')
print(f'Epochs P1+P2   : {PHASE1_EPOCHS} + {PHASE2_EPOCHS}')

In [ ]:
# Validasi semua path
errors = []

if IMAGE_DIR is None or not IMAGE_DIR.exists():
    errors.append(f'❌ Image dir tidak ada: {IMAGE_DIR}')
if ANNOT_JSON is None or not ANNOT_JSON.exists():
    errors.append(f'❌ Annotation path tidak ada: {ANNOT_JSON}')
if not SCRIPTS_DIR.exists():
    errors.append(f'❌ Scripts dir tidak ada: {SCRIPTS_DIR}')

if errors:
    raise RuntimeError('\n'.join(errors) + '\n\nPerbaiki path di cell konfigurasi di atas, lalu jalankan ulang.')

jpgs = []
for pattern in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG']:
    jpgs.extend(IMAGE_DIR.glob(pattern))
levels = {lvl: sum(1 for j in jpgs if j.name.startswith(lvl))
          for lvl in ['levle0', 'levle1', 'levle2', 'levle3']}

print('✅ Semua path valid')
print(f'   Total gambar : {len(jpgs)}')
print(f'   Device       : {DEVICE}')
print()
print('Distribusi kelas:')
for lvl, count in levels.items():
    bar = '█' * (count // 20)
    print(f'  {lvl}: {count:4d}  {bar}')


---
## 9. Preview Sample Gambar per Severity

In [ ]:
all_jpgs = []
for pattern in ['*.jpg', '*.JPG', '*.jpeg', '*.JPEG']:
    all_jpgs.extend(IMAGE_DIR.glob(pattern))
all_jpgs = sorted(all_jpgs)

samples = {}
for img_path in all_jpgs:
    for lvl in ['levle0', 'levle1', 'levle2', 'levle3']:
        if img_path.name.startswith(lvl) and lvl not in samples:
            samples[lvl] = img_path

if not samples:
    raise RuntimeError('Tidak ada sample levle0-levle3 ditemukan. Cek nama file gambar dataset.')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax in axes:
    ax.axis('off')
for ax, lvl in zip(axes, ['levle0', 'levle1', 'levle2', 'levle3']):
    path = samples.get(lvl)
    if path is None:
        ax.set_title(f'{lvl} tidak ada', fontsize=12)
        continue
    img = Image.open(path).resize((256, 256))
    ax.imshow(img)
    ax.set_title(lvl, fontsize=12)

plt.suptitle('Sample Gambar per Severity Level', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_ROOT / 'sample_images.png'), dpi=100)
plt.show()


---
## 10. Generate Refined Mask (Preprocessing)

Script `refine_masks_color.py` akan:
- Mendeteksi piksel lesi menggunakan Lab relative redness + HSV
- Menghasilkan refined mask yang lebih natural dari circle geometris
- Menyimpan gambar preprocessed (equalize + denoise + sharpen)
- Membuat `refined_manifest.csv` siap training

**Estimasi waktu: 5–15 menit untuk 1204 gambar di T4 GPU**

In [ ]:
if REFINED_MANIFEST.exists():
    df = pd.read_csv(REFINED_MANIFEST)
    print(f'ℹ️  Manifest sudah ada ({len(df)} baris) — skip generate.')
    print(df['split'].value_counts().to_string())
elif not REFINE_SCRIPT.exists():
    print('❌ Script tidak ditemukan:', REFINE_SCRIPT)
else:
    !python "{REFINE_SCRIPT}" \
      --image-dir       "{IMAGE_DIR}" \
      --annotations-json "{ANNOT_JSON}" \
      --output-dir      "{MASK_DIR}" \
      --apply-preprocessing \
      --max-images 0

    if REFINED_MANIFEST.exists():
        df = pd.read_csv(REFINED_MANIFEST)
        print(f'\n✅ Manifest dibuat: {len(df)} baris')
        print(df['split'].value_counts().to_string())
    else:
        print('❌ Gagal generate mask. Cek error di atas.')

In [ ]:
# Preview refined mask (3-panel: original | circle mask | refined mask)
viz_folder = MASK_DIR / 'visualizations'
viz_files  = sorted(viz_folder.glob('*.png'))[:8] if viz_folder.exists() else []

if viz_files:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, vf in zip(axes.flatten(), viz_files):
        img = Image.open(vf)
        ax.imshow(img)
        ax.set_title(vf.stem[:28], fontsize=7)
        ax.axis('off')
    plt.suptitle('Refined Mask — 3 panel: original | circle | refined', fontsize=12)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'refined_mask_preview.png'), dpi=80)
    plt.show()
else:
    print('Folder visualizations kosong atau belum ada.')

---
## 11. Training

Jalankan cell di bawah untuk melatih model **Proposal Hybrid ResNet-50 + MA-LDS**.

In [ ]:
# ══════════════════════════════════════════════════════════════
# OPTION A — Proposal Hybrid ResNet50 + MA-LDS (REKOMENDASI)
# Parameter training diatur di cell Konfigurasi di atas.
# ══════════════════════════════════════════════════════════════
if not TRAIN_SCRIPT_A.exists():
    print('❌ Script tidak ditemukan:', TRAIN_SCRIPT_A)
elif not REFINED_MANIFEST.exists():
    print('❌ refined_manifest.csv belum ada. Jalankan cell preprocessing dulu.')
else:
    !python "{TRAIN_SCRIPT_A}" \
      --manifest        "{REFINED_MANIFEST}" \
      --output-dir      "{OUTPUT_A}" \
      --image-size      {IMAGE_SIZE} \
      --batch-size      {BATCH_SIZE} \
      --phase1-epochs   {PHASE1_EPOCHS} \
      --phase2-epochs   {PHASE2_EPOCHS} \
      --device          {DEVICE} \
      --pretrained \
      --weighted-sampler \
      --num-workers     {NUM_WORKERS} \
      --seed 42

---
## 12. Monitor Training (jalankan sambil training berjalan)

In [ ]:
# Ganti ke OUTPUT_B / 'metrics.json' jika menjalankan Option B
METRICS_FILE = OUTPUT_A / 'metrics.json'

def plot_training_curves(metrics_path: Path):
    if not metrics_path.exists():
        print('metrics.json belum ada — training mungkin belum mulai.')
        return

    with open(metrics_path) as f:
        data = json.load(f)

    history = data.get('history', [])
    if not history:
        print('History kosong.')
        return

    epochs     = [h['epoch'] for h in history]
    train_loss = [h.get('train_loss', 0) for h in history]
    val_loss   = [h.get('val_loss', 0) for h in history]
    train_dice = [h.get('train_dice', 0) for h in history]
    val_dice   = [h.get('val_dice', 0) for h in history]
    train_acc  = [h.get('train_acc', 0) for h in history]
    val_acc    = [h.get('val_acc', 0) for h in history]
    train_kappa= [h.get('train_kappa', 0) for h in history]
    val_kappa  = [h.get('val_kappa', 0) for h in history]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    axes[0, 0].plot(epochs, train_loss, label='Train', color='steelblue')
    axes[0, 0].plot(epochs, val_loss,   label='Val',   color='orange')
    axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

    axes[0, 1].plot(epochs, train_dice, label='Train', color='green')
    axes[0, 1].plot(epochs, val_dice,   label='Val',   color='red')
    axes[0, 1].set_title('Dice Score'); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

    axes[1, 0].plot(epochs, train_acc, label='Train', color='steelblue')
    axes[1, 0].plot(epochs, val_acc,   label='Val',   color='orange')
    axes[1, 0].set_title('Accuracy'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

    axes[1, 1].plot(epochs, train_kappa, label='Train', color='purple')
    axes[1, 1].plot(epochs, val_kappa,   label='Val',   color='darkred')
    axes[1, 1].set_title("Cohen's Kappa"); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

    last = history[-1]
    fig.suptitle(
        f"Epoch {last['epoch']} | Phase: {last.get('phase', '?')} | "
        f"Val Dice: {last.get('val_dice', 0):.4f} | Val Acc: {last.get('val_acc', 0):.4f}",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'training_curves.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Epoch {last["epoch"]} | Val Dice: {last.get("val_dice",0):.4f} | Val Acc: {last.get("val_acc",0):.4f}')

plot_training_curves(METRICS_FILE)

---
## 13. Test Metrics

In [ ]:
def print_test_metrics(metrics_path: Path):
    if not metrics_path.exists():
        print('❌ metrics.json tidak ditemukan:', metrics_path)
        return

    with open(metrics_path) as f:
        data = json.load(f)

    tm  = data.get('test_metrics', {})
    cfg = data.get('config', {})

    print('=' * 45)
    print('       TEST METRICS — HASIL AKHIR')
    print('=' * 45)
    print(f"  Accuracy      : {tm.get('acc', 0):.4f}")
    print(f"  Cohen Kappa   : {tm.get('kappa', 0):.4f}")
    print(f"  Dice Score    : {tm.get('dice', 0):.4f}")
    print(f"  IoU           : {tm.get('iou', 0):.4f}")
    print('-' * 45)
    print(f"  Total Loss    : {tm.get('loss', 0):.4f}")
    print(f"  Seg Loss      : {tm.get('seg_loss', 0):.4f}")
    print(f"  Cls Loss      : {tm.get('cls_loss', 0):.4f}")
    print('=' * 45)
    print(f"  Best Val Loss : {data.get('best_val_loss', 0):.4f}")
    print(f"  Epochs (P1+P2): {cfg.get('phase1_epochs', '?')} + {cfg.get('phase2_epochs', '?')}")
    print(f"  Batch size    : {cfg.get('batch_size', '?')}")
    print(f"  Image size    : {cfg.get('image_size', '?')}")

print_test_metrics(OUTPUT_A / 'metrics.json')

---
## 14. Inference — Prediksi Test Set

In [ ]:
CHECKPOINT = OUTPUT_A / 'best_model.pt'

if not INFER_SCRIPT.exists():
    print('❌ Script tidak ditemukan:', INFER_SCRIPT)
elif not CHECKPOINT.exists():
    print('❌ Checkpoint tidak ditemukan:', CHECKPOINT)
    print('   Training harus selesai dulu.')
elif not REFINED_MANIFEST.exists():
    print('❌ refined_manifest.csv tidak ada.')
else:
    !python "{INFER_SCRIPT}" \
      --manifest    "{REFINED_MANIFEST}" \
      --checkpoint  "{CHECKPOINT}" \
      --output-dir  "{INFER_OUT}" \
      --split test \
      --image-size  320 \
      --threshold   0.5 \
      --device      {DEVICE} \
      --model-type  proposal

---
## 15. Visualisasi Hasil Segmentasi

In [ ]:
# Preview predicted mask vs ground truth
pred_folder = INFER_OUT / 'overlays'
if not pred_folder.exists():
    pred_folder = INFER_OUT

pred_files = sorted(pred_folder.glob('*.png'))[:8]

if pred_files:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, pf in zip(axes.flatten(), pred_files):
        img = Image.open(pf)
        ax.imshow(img)
        ax.set_title(pf.stem[:25], fontsize=8)
        ax.axis('off')
    plt.suptitle('Predicted Segmentation Mask — Test Set', fontsize=14)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'predicted_masks_preview.png'), dpi=100)
    plt.show()
else:
    print('Folder output inferensi kosong atau belum ada.')

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

infer_csv = INFER_OUT / 'testing_predictions.csv'

if not infer_csv.exists():
    print('testing_predictions.csv belum ada. Jalankan inferensi dulu.')
else:
    df_pred = pd.read_csv(infer_csv)
    print(f'Total prediksi: {len(df_pred)}')
    print(df_pred.head())

    if 'true_label_name' in df_pred.columns and 'pred_label_name' in df_pred.columns:
        labels = ['levle0', 'levle1', 'levle2', 'levle3']
        cm = confusion_matrix(df_pred['true_label_name'], df_pred['pred_label_name'], labels=labels)

        fig, ax = plt.subplots(figsize=(7, 6))
        disp = ConfusionMatrixDisplay(cm, display_labels=labels)
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title('Confusion Matrix — Test Set', fontsize=13)
        plt.tight_layout()
        plt.savefig(str(OUTPUT_ROOT / 'confusion_matrix.png'), dpi=100)
        plt.show()
    else:
        print('Kolom true_label_name/pred_label_name tidak ditemukan di testing_predictions.csv')


---
## 16. Rekap Semua Eksperimen

In [ ]:
result_files = list(TRAIN_DIR.rglob('experiment_results.csv'))
metrics_files = list(TRAIN_DIR.rglob('metrics.json'))

# Baca dari experiment_results.csv jika ada
if result_files:
    dfs = [pd.read_csv(f) for f in result_files]
    df_all = pd.concat(dfs, ignore_index=True)
    cols_show = ['run_name', 'encoder', 'epochs', 'test_acc', 'test_kappa', 'test_dice', 'test_iou']
    cols_exist = [c for c in cols_show if c in df_all.columns]
    print('=== REKAP SEMUA RUN ===')
    print(df_all[cols_exist].to_string(index=False))

# Fallback: baca dari masing-masing metrics.json
elif metrics_files:
    print('=== REKAP DARI metrics.json ===')
    rows = []
    for mf in metrics_files:
        with open(mf) as f:
            d = json.load(f)
        tm = d.get('test_metrics', {})
        rows.append({
            'run'  : mf.parent.name,
            'dice' : round(tm.get('dice', 0), 4),
            'iou'  : round(tm.get('iou', 0), 4),
            'acc'  : round(tm.get('acc', 0), 4),
            'kappa': round(tm.get('kappa', 0), 4),
        })
    print(pd.DataFrame(rows).to_string(index=False))

else:
    print('Belum ada hasil eksperimen. Training harus selesai dulu.')

---
## 17. Backup Checkpoint ke Drive

Output training sudah langsung disimpan ke Drive. Cell ini hanya untuk memastikan tidak ada checkpoint yang tertinggal di `/content/`.

In [ ]:
# Copy semua checkpoint dari /content/ ke Drive jika belum ada
checkpoints = list(Path('/content').rglob('best_model.pt'))
copied = 0

for ckpt in checkpoints:
    rel = ckpt.relative_to('/content')
    target = OUTPUT_ROOT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        shutil.copy2(ckpt, target)
        print(f'✅ Disalin: {ckpt} → {target}')
        copied += 1
    else:
        print(f'ℹ️  Sudah ada: {target}')

if not checkpoints:
    print('Tidak ada checkpoint di /content/')
else:
    print(f'\n{copied} file dicopy. Semua output tersimpan di: {OUTPUT_ROOT}')

---
## Troubleshooting

| Masalah | Solusi |
|---|---|
| `CUDA not available` | Runtime → Change runtime type → GPU |
| `CUDA out of memory` | Kurangi `BATCH_SIZE` ke `8` atau `IMAGE_SIZE` ke `224` |
| `NameError: Path` | Jalankan ulang cell **Import Python Libraries** (cell 4) |
| `ModuleNotFoundError` | Jalankan ulang cell Install Dependencies + Import |
| Dataset tidak terdownload | Pastikan link folder Drive bisa diakses publik, lalu jalankan ulang cell 5-6 |
| Scripts zip tidak ditemukan | Upload `acne_scripts.zip` ke `MyDrive/acne_project/` lalu jalankan ulang cell 7 |
| Sesi putus saat training | Jalankan ulang cell 2-8, lalu training dengan flag `--resume` |
| Training sangat lambat | Naikkan `NUM_WORKERS` ke `4`, atau upgrade ke Colab Pro |
| Dice masih 0 setelah training | Naikkan `focal_alpha` ke `0.85`, gunakan `--weighted-sampler` |

### Target metrik realistis (1204 gambar + T4 GPU)
| Metrik | Target |
|---|---|
| Dice | > 0.05 |
| IoU | > 0.03 |
| Accuracy | > 0.40 |
| Kappa | > 0.10 |
